<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab5-Graphs-with-NetworkX.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 5: Graphs with NetworkX

**Study Session 5: Graph Theory** · Davidson Open University

**In this lab you will:**
- build and draw graphs with the free NetworkX library;
- find degrees and check the handshaking lemma;
- write the adjacency matrix of a graph;
- decide whether a graph has an Euler path or circuit, and find shortest routes.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '5.1': {'h': '197921b2d6fbc3ae', 'hint': 'Use [d for v, d in H.degree()] and sort it with sorted(..., reverse=True).', 'ok': 'This list is called the degree sequence.'},
    '5.2': {'h': 'b4d541213f733299', 'hint': 'Number of edges = (sum of degrees) / 2.', 'ok': 'No drawing needed: that is the power of the handshaking lemma.'},
    '5.3': {'h': '4b5c37da224f9923', 'hint': 'Use the nodelist in the order 1 to 6 so that row 1 is vertex 1.', 'ok': 'The matrix is symmetric, and each row adds up to the degree of that vertex.'},
    '5.4': {'h': '080fb1cf7ebc6516', 'hint': 'Build the graph and call .number_of_edges().', 'ok': '10 × 9 / 2 = 45, which matches the formula.'},
    '5.5': {'h': 'ff202f34ce73f34e', 'hint': 'Count odd degrees with sum(1 for v, d in G.degree() if d % 2 == 1). 0 odd vertices → circuit; 2 → path; anything else → neither.', 'ok': 'This is the theorem Euler used to solve the Königsberg bridge problem in 1736.', 'args': [(__import__("networkx").cycle_graph(5),), (__import__("networkx").path_graph(4),), (__import__("networkx").complete_graph(4),), (__import__("networkx").complete_graph(5),), (__import__("networkx").star_graph(3),)]},
    '5.6': {'h': '316af2424d6d0632', 'hint': "Do not forget weight='weight', otherwise NetworkX counts edges instead of kilometres.", 'ok': 'Map apps use the same idea (Dijkstra’s algorithm) on millions of roads.'},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
A **graph** G = (V, E) has a set of **vertices** V and a set of **edges** E, each joining two vertices. **NetworkX** is a free Python library for graphs. Run the cell to build and draw a graph.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_edges_from([('a', 'b'), ('a', 'c'), ('b', 'c'), ('c', 'd'), ('d', 'e'), ('c', 'e')])
nx.draw(G, with_labels=True, node_color='#cfe2ff', node_size=700)
plt.show()
print('Vertices:', sorted(G.nodes()))
print('Number of edges:', G.number_of_edges())
print('Degrees:', dict(G.degree()))

The **degree** of a vertex is the number of edges that meet there. The **handshaking lemma** says the degrees always add up to **twice** the number of edges, because each edge has two ends. **Predict** the sum of the degrees above, then run the cell.

In [ ]:
total = sum(d for v, d in G.degree())
print('Sum of degrees =', total, '   2 × number of edges =', 2 * G.number_of_edges())

## Part B – Practise (about 20 minutes)
Use this graph H for B1 to B3 (run the cell to draw it):

In [ ]:
H = nx.Graph([(1, 2), (1, 3), (1, 4), (2, 3), (3, 4), (4, 5), (5, 6), (4, 6)])
nx.draw(H, with_labels=True, node_color='#d1e7dd', node_size=600)
plt.show()

In [ ]:
answer_5_1 = ...   # the list of degrees of H, from largest to smallest

In [ ]:
check('5.1', answer_5_1)

**B2.** A graph has degree sequence 4, 3, 3, 2, 2, 2. Without drawing it, use the handshaking lemma to find its number of edges.

In [ ]:
answer_5_2 = ...

In [ ]:
check('5.2', answer_5_2)

**B3.** The **adjacency matrix** A has a 1 in row i, column j when vertices i and j are joined, and 0 otherwise. Use `nx.to_numpy_array(H, nodelist=[1, 2, 3, 4, 5, 6], dtype=int)` and turn it into a list with `.tolist()`.

In [ ]:
answer_5_3 = ...

In [ ]:
check('5.3', answer_5_3)

**B4. Complete graphs.** In the complete graph Kₙ every pair of vertices is joined. Use `nx.complete_graph(10)` to find the number of edges of K₁₀, and compare it with the formula n(n − 1)/2.

In [ ]:
answer_5_4 = ...

In [ ]:
check('5.4', answer_5_4)

## Part C – Build a tool (about 20 minutes)
### C1. Euler paths and circuits
For a **connected** graph:
- an **Euler circuit** (using every edge once, ending where it started) exists exactly when **every** vertex has even degree;
- an **Euler path** (using every edge once, not returning to the start) exists exactly when **exactly two** vertices have odd degree.

Complete `euler_status(G)` so that it returns `'circuit'`, `'path'` or `'neither'`. (The first two lines already handle disconnected graphs, which have neither.) *Hint:* `nx.is_connected(G)` and count the vertices with odd degree.

In [ ]:
def euler_status(G):
    if not nx.is_connected(G):
        return 'neither'
    odd = ...        # YOUR CODE: the number of vertices with odd degree
    # YOUR CODE: return 'circuit', 'path' or 'neither'
    ...

print(euler_status(G))

In [ ]:
check('5.5', euler_status)

### C2. Shortest routes on a weighted graph
Give each edge a **weight**, for example a distance in km. NetworkX can find the shortest route. The map below shows some places in Ibadan (distances are made up for the exercise).

In [ ]:
W = nx.Graph()
W.add_weighted_edges_from([
    ('Bodija', 'UI', 3), ('Bodija', 'Dugbe', 6), ('UI', 'Ojoo', 4), ('UI', 'Mokola', 4),
    ('Dugbe', 'Mokola', 3), ('Ojoo', 'Mokola', 7), ('Dugbe', 'Challenge', 5), ('Mokola', 'Challenge', 6)])
pos = nx.spring_layout(W, seed=3)
nx.draw(W, pos, with_labels=True, node_color='#fff3cd', node_size=1400, font_size=8)
nx.draw_networkx_edge_labels(W, pos, edge_labels=nx.get_edge_attributes(W, 'weight'))
plt.show()

**C2.** Find the shortest route from **Ojoo** to **Challenge** and its length. Use `nx.shortest_path(W, 'Ojoo', 'Challenge', weight='weight')` and `nx.shortest_path_length(...)` with the same inputs. Store the answer as a tuple `(route, length)`.

In [ ]:
route = ...
length = ...
answer_5_6 = (route, length)
print(answer_5_6)

In [ ]:
check('5.6', answer_5_6 if ... not in answer_5_6 else ...)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 5.** Model a small network you know (5–8 bus stops, or friends who message each other). Build it in the cell, draw it, print the degrees and the adjacency matrix, and use `euler_status`. You can also draw graphs in the free online graph editor linked on the lab page.

In [ ]:
# D1: your own network
my_graph = nx.Graph()
# my_graph.add_edges_from([...])

**D2. Reflect.** Answer in one or two sentences each:
1. Why can a graph never have exactly one vertex of odd degree?
2. Your graph in D1: does it have an Euler path or circuit? What would you add or remove to change that?
3. Name one real problem in Nigeria that could be modelled as a shortest-route problem.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 5 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.